                    TRANSFORMER
                         │
                         ↓
                    Embeddings
                         ↓
              Positional Information
                         ↓
              ┌────────────────────┐
              │ Transformer Block  │
              │                    │
              │ Multi-Head         │
              │ Self-Attention     │
              │       ↓            │
              │ Residual + Norm    │
              │       ↓            │
              │ Feed Forward       │
              │       ↓            │
              │ Residual + Norm    │
              └─────────┬──────────┘
                        ↓
                 Repeat N times
                        ↓
                   Final Layer
                        ↓
                 Next-token output

In [1]:
text = """
i love cats
i love dogs
i like cats
i like dogs
cats are cute
dogs are cute
cats are animals
dogs are animals
i love animals
"""

# Tokenization

In [2]:
tokens = text.lower().split()

print(tokens)

['i', 'love', 'cats', 'i', 'love', 'dogs', 'i', 'like', 'cats', 'i', 'like', 'dogs', 'cats', 'are', 'cute', 'dogs', 'are', 'cute', 'cats', 'are', 'animals', 'dogs', 'are', 'animals', 'i', 'love', 'animals']


# Vocabulary

In [3]:
vocab = sorted(set(tokens))

print(vocab)

['animals', 'are', 'cats', 'cute', 'dogs', 'i', 'like', 'love']


# ID Mapping

In [4]:
stoi = {word: i for i, word in enumerate(vocab)}
itos = {i: word for word, i in stoi.items()}

print(stoi)
print(itos)

{'animals': 0, 'are': 1, 'cats': 2, 'cute': 3, 'dogs': 4, 'i': 5, 'like': 6, 'love': 7}
{0: 'animals', 1: 'are', 2: 'cats', 3: 'cute', 4: 'dogs', 5: 'i', 6: 'like', 7: 'love'}


# Convert our entire Dataset to Id's

In [5]:
encoded = [stoi[word] for word in tokens]

print(encoded)
print(len(encoded))

[5, 7, 2, 5, 7, 4, 5, 6, 2, 5, 6, 4, 2, 1, 3, 4, 1, 3, 2, 1, 0, 4, 1, 0, 5, 7, 0]
27


# Training Sequences

In [6]:
seq_length = 4

X = []
y = []

for i in range(len(encoded) - seq_length):
    X.append(encoded[i:i + seq_length])
    y.append(encoded[i + 1:i + seq_length + 1])

# Convert to tensors

In [7]:
import torch

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print(X.shape)
print(y.shape)

torch.Size([23, 4])
torch.Size([23, 4])


# **Embedding Layer**

**Token IDs → Embeddings**

In [8]:
import torch
import torch.nn as nn

vocab_size = len(vocab)
embedding_dim = 8

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim
)

In [9]:
print(embedding.weight)

Parameter containing:
tensor([[-1.6538,  1.4861,  0.7826,  1.1601, -1.5306,  0.2849, -0.3881,  0.7829],
        [-0.3314, -0.8314,  0.4647,  0.6557,  0.8403,  0.7118, -0.8646,  0.3176],
        [-1.5871, -0.0081,  0.2245, -0.9505, -0.3781,  0.1586,  1.6013, -1.4068],
        [-2.0875, -0.7950,  0.0881,  0.1993,  1.2225, -0.4469, -2.9806, -0.5573],
        [ 1.7580, -1.4804, -1.4223, -0.0486, -1.0617,  0.6127, -0.6444,  1.0598],
        [ 0.5339, -0.1966, -0.7896,  0.6685, -0.3373, -1.5979, -1.4106, -0.4161],
        [-0.0405,  1.1969, -0.7610, -0.2812,  0.6527,  0.4224, -0.3521, -0.4177],
        [ 2.4306,  0.2666, -0.3363,  1.2535,  0.1646, -0.2260, -0.4036,  0.9890]],
       requires_grad=True)


In [10]:
x = torch.tensor([[5, 7, 2, 5]])

embedded = embedding(x)
print(embedded)

print(x.shape)
print(embedded.shape)

tensor([[[ 0.5339, -0.1966, -0.7896,  0.6685, -0.3373, -1.5979, -1.4106,
          -0.4161],
         [ 2.4306,  0.2666, -0.3363,  1.2535,  0.1646, -0.2260, -0.4036,
           0.9890],
         [-1.5871, -0.0081,  0.2245, -0.9505, -0.3781,  0.1586,  1.6013,
          -1.4068],
         [ 0.5339, -0.1966, -0.7896,  0.6685, -0.3373, -1.5979, -1.4106,
          -0.4161]]], grad_fn=<EmbeddingBackward0>)
torch.Size([1, 4])
torch.Size([1, 4, 8])


**Positional Embedding**

In [11]:
max_seq_length = 4

position_embedding = nn.Embedding(
    max_seq_length,
    embedding_dim
)

In [12]:
positions = torch.arange(4)

print(positions)

tensor([0, 1, 2, 3])


In [13]:
pos_emb = position_embedding(positions)

print(pos_emb.shape)

torch.Size([4, 8])


**Add token + position embeddings**

In [14]:
x_emb = embedded + pos_emb

# **Self Attention**

In [15]:
import torch
import torch.nn as nn

attention_dim = embedding_dim

Wq = nn.Linear(
    embedding_dim,
    attention_dim
)

Wk = nn.Linear(
    embedding_dim,
    attention_dim
)

Wv = nn.Linear(
    embedding_dim,
    attention_dim
)

Q = Wq(x_emb)

K = Wk(x_emb)

V = Wv(x_emb)


print("\nQ shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)


Q shape: torch.Size([1, 4, 8])
K shape: torch.Size([1, 4, 8])
V shape: torch.Size([1, 4, 8])


In [16]:
scores = torch.matmul(
    Q,
    K.transpose(-2, -1)
)

print("\nAttention scores shape:")
print(scores.shape)

print("\nRaw attention scores:")
print(scores)



Attention scores shape:
torch.Size([1, 4, 4])

Raw attention scores:
tensor([[[-3.2848,  0.1323,  1.0350, -3.5537],
         [ 3.0758, -2.7720, -4.3010,  0.1897],
         [ 9.8516, -3.9883, -4.0270,  4.1140],
         [ 0.9718, -2.8836, -2.2157, -2.4313]]], grad_fn=<UnsafeViewBackward0>)


**Scaling**

In [17]:
d_k = K.size(-1)

scores = scores / (d_k ** 0.5)

print("\nScaled attention scores:")
print(scores)


Scaled attention scores:
tensor([[[-1.1614,  0.0468,  0.3659, -1.2564],
         [ 1.0875, -0.9801, -1.5206,  0.0671],
         [ 3.4831, -1.4101, -1.4238,  1.4545],
         [ 0.3436, -1.0195, -0.7834, -0.8596]]], grad_fn=<DivBackward0>)


**Casual Mask**

In [18]:
mask = torch.tril(
    torch.ones(
        seq_length,
        seq_length
    )
)

print("\nCausal mask:")
print(mask)


Causal mask:
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


In [19]:
scores = scores.masked_fill(
    mask == 0,
    float("-inf")
)

print("\nMasked scores:")
print(scores)


Masked scores:
tensor([[[-1.1614,    -inf,    -inf,    -inf],
         [ 1.0875, -0.9801,    -inf,    -inf],
         [ 3.4831, -1.4101, -1.4238,    -inf],
         [ 0.3436, -1.0195, -0.7834, -0.8596]]], grad_fn=<MaskedFillBackward0>)


**Softmax**

In [20]:
import torch.nn.functional as F

attention_weights = F.softmax(
    scores,
    dim=-1
)

print("\nAttention weights:")
print(attention_weights)

print("\nAttention weights shape:")
print(attention_weights.shape)


Attention weights:
tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.8877, 0.1123, 0.0000, 0.0000],
         [0.9853, 0.0074, 0.0073, 0.0000],
         [0.5319, 0.1361, 0.1723, 0.1597]]], grad_fn=<SoftmaxBackward0>)

Attention weights shape:
torch.Size([1, 4, 4])


**Multiply Attention Weight x Values**

In [21]:
attention_output = torch.matmul(
    attention_weights,
    V
)

print("\nSelf-attention output:")
print(attention_output)

print("\nSelf-attention output shape:")
print(attention_output.shape)


Self-attention output:
tensor([[[ 0.9698, -0.2232,  0.7797,  0.4015, -0.4912,  0.6293,  1.2435,
           0.6512],
         [ 0.8918, -0.1708,  0.5670,  0.3577, -0.6151,  0.5051,  1.0198,
           0.5949],
         [ 0.9491, -0.2121,  0.7587,  0.3915, -0.4929,  0.6251,  1.2207,
           0.6500],
         [ 0.4502,  0.1833,  0.4491,  0.2173, -0.5025,  0.5898,  0.6997,
           0.7520]]], grad_fn=<UnsafeViewBackward0>)

Self-attention output shape:
torch.Size([1, 4, 8])


In [22]:
print("\n" + "=" * 60)
print("FINAL SELF-ATTENTION PIPELINE")
print("=" * 60)

print("Input token IDs:")
print(x)

print("\nQ:")
print(Q)

print("\nK:")
print(K)

print("\nV:")
print(V)

print("\nAttention weights:")
print(attention_weights)

print("\nFinal self-attention output:")
print(attention_output)


FINAL SELF-ATTENTION PIPELINE
Input token IDs:
tensor([[5, 7, 2, 5]])

Q:
tensor([[[ 1.7033, -1.7588, -0.1421, -0.0610, -0.2135,  0.2924,  0.7095,
          -0.3623],
         [-1.2214,  0.3492,  0.7699,  0.2784,  0.7397,  1.5872,  1.4206,
           0.4703],
         [-0.4800, -0.0902,  0.8252, -2.1043,  1.7173, -0.5670,  0.7982,
           1.9850],
         [ 0.3864, -1.3064, -0.7744, -1.2545,  1.0135,  0.5630,  0.7916,
           0.4806]]], grad_fn=<ViewBackward0>)

K:
tensor([[[-1.3603, -0.2155,  1.7359, -0.7754,  1.1777, -0.8675,  0.0175,
           1.8069],
         [-1.0682, -1.5994,  1.6288, -0.3736, -1.4791, -0.0432, -2.0270,
          -1.3311],
         [ 2.4194,  1.5845, -0.7942,  0.9818,  0.4318, -0.7830, -0.2896,
          -0.4815],
         [-0.4739,  0.7298,  1.0223,  0.5156,  0.4522, -1.3133, -0.3718,
           1.4959]]], grad_fn=<ViewBackward0>)

V:
tensor([[[ 0.9698, -0.2232,  0.7797,  0.4015, -0.4912,  0.6293,  1.2435,
           0.6512],
         [ 0.2749,  0.2436

# **Multi-Head Self-Attention**

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):

    def __init__(self, embedding_dim, num_heads):

        super().__init__()

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads

        # Each head gets part of the embedding dimensions
        self.head_dim = embedding_dim // num_heads

        assert embedding_dim % num_heads == 0, \
            "embedding_dim must be divisible by num_heads"

        # Q, K, V projections
        self.Wq = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        self.Wk = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        self.Wv = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        # Final projection after combining heads
        self.out_proj = nn.Linear(
            embedding_dim,
            embedding_dim
        )


    def forward(self, x):

        batch_size, seq_length, embedding_dim = x.shape

        # ------------------------------------------------
        # 1. Create Q, K, V
        # ------------------------------------------------

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)


        # ------------------------------------------------
        # 2. Split into multiple heads
        # ------------------------------------------------

        Q = Q.view(
            batch_size,
            seq_length,
            self.num_heads,
            self.head_dim
        )

        K = K.view(
            batch_size,
            seq_length,
            self.num_heads,
            self.head_dim
        )

        V = V.view(
            batch_size,
            seq_length,
            self.num_heads,
            self.head_dim
        )


        # ------------------------------------------------
        # 3. Move heads before sequence dimension
        # ------------------------------------------------

        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)


        # ------------------------------------------------
        # 4. Attention scores
        # ------------------------------------------------

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )


        # ------------------------------------------------
        # 5. Scale
        # ------------------------------------------------

        scores = scores / (self.head_dim ** 0.5)


        # ------------------------------------------------
        # 6. Causal mask
        # ------------------------------------------------

        mask = torch.tril(
            torch.ones(
                seq_length,
                seq_length,
                device=x.device
            )
        )

        scores = scores.masked_fill(
            mask == 0,
            float("-inf")
        )


        # ------------------------------------------------
        # 7. Softmax
        # ------------------------------------------------

        attention_weights = F.softmax(
            scores,
            dim=-1
        )


        # ------------------------------------------------
        # 8. Weighted sum of Values
        # ------------------------------------------------

        attention_output = torch.matmul(
            attention_weights,
            V
        )


        # ------------------------------------------------
        # 9. Combine heads
        # ------------------------------------------------

        attention_output = attention_output.transpose(1, 2)

        attention_output = attention_output.contiguous().view(
            batch_size,
            seq_length,
            self.embedding_dim
        )


        # ------------------------------------------------
        # 10. Final projection
        # ------------------------------------------------

        output = self.out_proj(
            attention_output
        )

        return output

# Residual Connection

In [24]:
class AttentionWithResidualNorm(nn.Module):

    def __init__(self, embedding_dim, num_heads):

        super().__init__()

        self.attention = MultiHeadAttention(
            embedding_dim,
            num_heads
        )

        self.norm = nn.LayerNorm(
            embedding_dim
        )


    def forward(self, x):

        # Keep original input
        residual = x

        # Self-attention
        attention_output = self.attention(x)

        # Residual connection
        x = residual + attention_output

        # Layer normalization
        x = self.norm(x)

        return x

In [25]:
embedding_dim = 8
num_heads = 2

attention_block = AttentionWithResidualNorm(
    embedding_dim,
    num_heads
)

output = attention_block(x_emb)

print("Input shape:")
print(x_emb.shape)

print("\nOutput shape:")
print(output.shape)

Input shape:
torch.Size([1, 4, 8])

Output shape:
torch.Size([1, 4, 8])


# Feed Forward Neural Network

In [26]:
class FeedForward(nn.Module):

    def __init__(self, embedding_dim):

        super().__init__()

        self.network = nn.Sequential(

            # Expand
            nn.Linear(
                embedding_dim,
                embedding_dim * 4
            ),

            # Non-linearity
            nn.GELU(),

            # Compress back
            nn.Linear(
                embedding_dim * 4,
                embedding_dim
            )
        )


    def forward(self, x):

        return self.network(x)

In [27]:
ffn = FeedForward(embedding_dim)

ffn_output = ffn(output)

print("Input to FFN:")
print(output.shape)

print("\nOutput from FFN:")
print(ffn_output.shape)

Input to FFN:
torch.Size([1, 4, 8])

Output from FFN:
torch.Size([1, 4, 8])


# Transformer Block

In [28]:
class TransformerBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads):

        super().__init__()

        # Multi-head self-attention
        self.attention = MultiHeadAttention(
            embedding_dim,
            num_heads
        )

        # Feed-forward network
        self.ffn = FeedForward(
            embedding_dim
        )

        # Layer normalizations
        self.norm1 = nn.LayerNorm(
            embedding_dim
        )

        self.norm2 = nn.LayerNorm(
            embedding_dim
        )


    def forward(self, x):

        # ==========================================
        # 1. Self-Attention
        # ==========================================

        attention_output = self.attention(x)


        # ==========================================
        # 2. Residual + LayerNorm
        # ==========================================

        x = self.norm1(
            x + attention_output
        )


        # ==========================================
        # 3. Feed Forward
        # ==========================================

        ffn_output = self.ffn(x)


        # ==========================================
        # 4. Residual + LayerNorm
        # ==========================================

        x = self.norm2(
            x + ffn_output
        )


        return x

In [29]:
embedding_dim = 8
num_heads = 2

transformer_block = TransformerBlock(
    embedding_dim,
    num_heads
)

output = transformer_block(x_emb)

print("Input:")
print(x_emb.shape)

print("\nOutput:")
print(output.shape)

Input:
torch.Size([1, 4, 8])

Output:
torch.Size([1, 4, 8])


# Stacking Multiple Layers

In [30]:
class TransformerStack(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_heads,
        num_layers
    ):

        super().__init__()

        self.blocks = nn.ModuleList([

            TransformerBlock(
                embedding_dim,
                num_heads
            )

            for _ in range(num_layers)
        ])


    def forward(self, x):

        for block in self.blocks:

            x = block(x)

        return x

In [31]:
lm_head = nn.Linear(
    embedding_dim,
    vocab_size
)

logits = lm_head(output)

**Softmax**

In [32]:
probabilities = F.softmax(
    logits,
    dim=-1
)

# Tiny GPT

In [33]:
class TinyGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_heads,
        num_layers,
        max_seq_length
    ):

        super().__init__()

        # -----------------------------
        # Token Embedding
        # -----------------------------

        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        # -----------------------------
        # Positional Embedding
        # -----------------------------

        self.position_embedding = nn.Embedding(
            max_seq_length,
            embedding_dim
        )

        # -----------------------------
        # Transformer Blocks
        # -----------------------------

        self.blocks = nn.ModuleList([

            TransformerBlock(
                embedding_dim,
                num_heads
            )

            for _ in range(num_layers)

        ])

        # -----------------------------
        # Final LayerNorm
        # -----------------------------

        self.final_norm = nn.LayerNorm(
            embedding_dim
        )

        # -----------------------------
        # Language Model Head
        # -----------------------------

        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size
        )


    def forward(self, x):

        batch_size, seq_length = x.shape

        # -----------------------------
        # Token embeddings
        # -----------------------------

        token_emb = self.token_embedding(x)

        # -----------------------------
        # Position embeddings
        # -----------------------------

        positions = torch.arange(
            seq_length,
            device=x.device
        )

        pos_emb = self.position_embedding(
            positions
        )

        # -----------------------------
        # Combine embeddings
        # -----------------------------

        x = token_emb + pos_emb

        # -----------------------------
        # Transformer blocks
        # -----------------------------

        for block in self.blocks:

            x = block(x)

        # -----------------------------
        # Final normalization
        # -----------------------------

        x = self.final_norm(x)

        # -----------------------------
        # Language model head
        # -----------------------------

        logits = self.lm_head(x)

        return logits

**Create the Model**

In [34]:
vocab_size = len(vocab)

embedding_dim = 8

num_heads = 2

num_layers = 4

max_seq_length = 4

model = TinyGPT(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    max_seq_length=max_seq_length
)

print(model)

TinyGPT(
  (token_embedding): Embedding(8, 8)
  (position_embedding): Embedding(4, 8)
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (attention): MultiHeadAttention(
        (Wq): Linear(in_features=8, out_features=8, bias=True)
        (Wk): Linear(in_features=8, out_features=8, bias=True)
        (Wv): Linear(in_features=8, out_features=8, bias=True)
        (out_proj): Linear(in_features=8, out_features=8, bias=True)
      )
      (ffn): FeedForward(
        (network): Sequential(
          (0): Linear(in_features=8, out_features=32, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=32, out_features=8, bias=True)
        )
      )
      (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
    )
  )
  (final_norm): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=8, out_features=8, bias=True)
)


In [36]:
sample = X[0].unsqueeze(0)

print("Sample:")
print(sample)

print("Shape:")
print(sample.shape)

logits = model(sample)

print("Logits shape:")
print(logits.shape)

Sample:
tensor([[5, 7, 2, 5]])
Shape:
torch.Size([1, 4])
Logits shape:
torch.Size([1, 4, 8])


In [38]:
# Get the logits for the last token position
last_logits = logits[:, -1, :]

print("Last token logits:")
print(last_logits)

print("\nShape:")
print(last_logits.shape)

Last token logits:
tensor([[ 0.1984, -0.3494, -1.1946, -0.3147,  0.7880,  0.2808, -0.3672, -0.7265]],
       grad_fn=<SelectBackward0>)

Shape:
torch.Size([1, 8])


In [39]:
predicted_id = torch.argmax(
    last_logits,
    dim=-1
)

print("Predicted token ID:")
print(predicted_id)

predicted_word = itos[predicted_id.item()]

print("\nPredicted word:")
print(predicted_word)

Predicted token ID:
tensor([4])

Predicted word:
dogs


In [40]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 500

for epoch in range(epochs):

    # Forward pass
    logits = model(X)

    # Reshape logits
    logits = logits.view(
        -1,
        vocab_size
    )

    # Reshape targets
    targets = y.view(
        -1
    )

    # Calculate loss
    loss = criterion(
        logits,
        targets
    )

    # Clear old gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update model weights
    optimizer.step()

    # Print progress
    if (epoch + 1) % 50 == 0:

        print(
            f"Epoch {epoch + 1}/{epochs}, "
            f"Loss: {loss.item():.4f}"
        )

Epoch 50/500, Loss: 1.7336
Epoch 100/500, Loss: 1.2917
Epoch 150/500, Loss: 1.0262
Epoch 200/500, Loss: 0.8209
Epoch 250/500, Loss: 0.6520
Epoch 300/500, Loss: 0.5245
Epoch 350/500, Loss: 0.4468
Epoch 400/500, Loss: 0.3959
Epoch 450/500, Loss: 0.3646
Epoch 500/500, Loss: 0.3457


In [42]:
model.eval()

while True:

    sentence = input("\nEnter your sentence (type 'exit' to quit): ")

    if sentence.lower() == "exit":
        break

    # Convert sentence into tokens
    words = sentence.lower().split()

    # Check vocabulary
    unknown_words = [word for word in words if word not in stoi]

    if unknown_words:
        print("Unknown words:", unknown_words)
        print("Please use only these words:")
        print(list(stoi.keys()))
        continue

    # Convert words → token IDs
    token_ids = [stoi[word] for word in words]

    # Convert to tensor
    input_tensor = torch.tensor(
        [token_ids],
        dtype=torch.long
    )

    # Check sequence length
    if input_tensor.size(1) > max_seq_length:
        print(f"Please enter at most {max_seq_length} words.")
        continue

    # Prediction
    with torch.no_grad():

        logits = model(input_tensor)

        # Take the last token's prediction
        last_logits = logits[:, -1, :]

        # Get highest-scoring token
        predicted_id = torch.argmax(
            last_logits,
            dim=-1
        ).item()

    # Convert ID → word
    predicted_word = itos[predicted_id]

    print("Input:", sentence)
    print("Predicted next word:", predicted_word)


Enter your sentence (type 'exit' to quit): i love
Input: i love
Predicted next word: dogs

Enter your sentence (type 'exit' to quit): cats are
Input: cats are
Predicted next word: cute

Enter your sentence (type 'exit' to quit): dogs are
Input: dogs are
Predicted next word: cute

Enter your sentence (type 'exit' to quit): i 
Input: i 
Predicted next word: like

Enter your sentence (type 'exit' to quit): i
Input: i
Predicted next word: like

Enter your sentence (type 'exit' to quit): i love
Input: i love
Predicted next word: dogs

Enter your sentence (type 'exit' to quit): exit
